In [1]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

c:\Users\kshit\OneDrive\Desktop\CodingNinjasAICourse\Rag\Vector Databases\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1) Create 100 one-liner texts
texts = [
    "AI models are getting larger every year",
    "Coffee prices fluctuate with global demand",
    "Python loops make repetitive tasks easier",
    "SQL joins merge data across related tables",
    "The best time to email customers is afternoon",
    "Vector databases improve semantic search",
    "Transfer learning reduces training time",
    "Open source drives faster innovation",
    "Cosine similarity measures text closeness",
    "Apple launches new devices every September",
    "Pandas groupby aggregates large datasets",
    "Feature engineering improves model accuracy",
    "Normalization helps models converge faster",
    "Random forests combine many decision trees",
    "Gradient boosting refines weak learners",
    "K-means clustering groups similar points",
    "Dimensionality reduction speeds up training",
    "Visualization helps explain complex data",
    "Streaming data needs real-time processing",
    "Outlier detection prevents model bias",
    "Cross-validation improves generalization",
    "Hyperparameter tuning optimizes accuracy",
    "Batch processing handles large datasets",
    "Cloud storage scales automatically",
    "Data pipelines automate ETL workflows",
    "Machine learning powers recommendation systems",
    "Text embeddings convert words into vectors",
    "Image embeddings capture visual similarity",
    "Speech recognition converts sound to text",
    "Transformers replaced RNNs in NLP",
    "Attention mechanism focuses on context",
    "Reinforcement learning learns by reward",
    "Synthetic data helps balance datasets",
    "Time series forecasting predicts demand",
    "Business intelligence helps decision making",
    "Interactive dashboards improve insights",
    "A/B testing measures campaign success",
    "Customer segmentation improves marketing",
    "Fraud detection protects financial systems",
    "Personalization improves customer experience",
    "Deep learning requires large datasets",
    "Model evaluation ensures reliability",
    "Explainability builds model trust",
    "Ethics in AI ensures responsible innovation",
    "Data governance ensures data quality",
    "API integration connects multiple systems",
    "Microservices improve scalability",
    "Docker containers simplify deployment",
    "CI/CD pipelines automate software releases",
    "Serverless computing optimizes cost",
    "Edge AI brings inference closer to devices",
    "Federated learning protects privacy",
    "Prompt engineering improves LLM responses",
    "Tokenization breaks text into subwords",
    "Embeddings capture semantic meaning",
    "Vector search retrieves similar content",
    "RAG combines retrieval with generation",
    "LangChain simplifies LLM orchestration",
    "Agents coordinate multiple AI tasks",
    "Pinecone indexes large vector datasets",
    "Chroma provides lightweight vector storage",
    "Faiss accelerates approximate nearest neighbor search",
    "Sentence transformers create text embeddings",
    "Fine-tuning customizes pretrained models",
    "Quantization reduces model size",
    "Distillation transfers knowledge to smaller models",
    "LoRA fine-tunes large models efficiently",
    "RLHF aligns models with human feedback",
    "GPT models use autoregressive generation",
    "BERT uses bidirectional attention",
    "LLMs understand context and nuance",
    "OpenAI advances general-purpose AI",
    "Google trains foundation models at scale",
    "Meta focuses on open research models",
    "Anthropic emphasizes safety and alignment",
    "NVIDIA optimizes GPUs for deep learning",
    "Transformer architecture scales efficiently",
    "Self-attention connects distant tokens",
    "Batch normalization stabilizes learning",
    "Dropout prevents overfitting",
    "ReLU activation speeds convergence",
    "Adam optimizer adapts learning rates",
    "Learning rate scheduling improves training",
    "Data augmentation improves robustness",
    "Model checkpoints save progress",
    "Early stopping prevents overtraining",
    "Evaluation metrics guide improvement",
    "Precision and recall measure performance",
    "ROC curve visualizes trade-offs",
    "Confusion matrix summarizes predictions",
    "F1 score balances precision and recall",
    "AUC measures classification quality",
    "Regression minimizes squared errors",
    "Classification predicts categorical outcomes",
    "Clustering groups unlabeled data",
    "Anomaly detection identifies unusual patterns",
    "Dimensionality reduction simplifies data visualization",
    "Principal component analysis reduces redundancy",
    "t-SNE projects data into 2D space",
    "UMAP preserves local relationships",
    "Word2Vec learns from co-occurrence statistics",
    "Doc2Vec represents full sentences as vectors",
    "TF-IDF weighs word importance in documents",
    "BM25 scores document relevance in search engines",
]

In [3]:
# STEP 1 — initialize vectordb (in-memory) + config

# simple doc/vec stores
doc_db = {}   # id -> raw text
vec_db = {}   # id -> vector (optional mirror)

# embedding model
model_name = "sentence-transformers/paraphrase-MiniLM-L6-v2"
model = SentenceTransformer(model_name)
dim = model.get_sentence_embedding_dimension()  # e.g., 384 for MiniLM

# HNSW params
M   = 16     # max neighbors per node
efC = 100    # build effort
efS = 64     # search effort
metric = faiss.METRIC_INNER_PRODUCT  # use cosine via L2-normalization + IP

In [ ]:
# STEP 2 — create index (if it doesn't exist)

# Base HNSW index; some FAISS builds don't accept metric in ctor → fallback
base = faiss.IndexHNSWFlat(dim, M, metric)

base.hnsw.efConstruction = efC
base.hnsw.efSearch = efS

# Wrap with ID map to use your own ids
index = faiss.IndexIDMap2(base)

In [5]:
base

<faiss.swigfaiss_avx2.IndexHNSWFlat; proxy of <Swig Object of type 'faiss::IndexHNSWFlat *' at 0x000002065A0919B0> >

In [6]:
index

<faiss.swigfaiss_avx2.IndexIDMap2; proxy of <Swig Object of type 'faiss::IndexIDMap2Template< faiss::Index > *' at 0x000002065A05FA20> >

In [7]:
# STEP 4 — organize documents as [{id, doc}]

doc_records = [{"id": i, "doc": txt} for i, txt in enumerate(texts)]
for r in doc_records:
    doc_db[r["id"]] = r["doc"]

len(doc_records), doc_records[0]

(104, {'id': 0, 'doc': 'AI models are getting larger every year'})

In [8]:
# STEP 5 — create embeddings and organize as [{id, [embedding]}]

X = model.encode([r["doc"] for r in doc_records], convert_to_numpy=True).astype(np.float32)

# normalize so IP == cosine (and L2 distance ranking is equivalent on unit vectors)
faiss.normalize_L2(X)

ids = np.array([r["id"] for r in doc_records], dtype=np.int64)

# optional mirror
for pid, vec in zip(ids, X):
    vec_db[int(pid)] = vec

len(ids), X.shape

(104, (104, 384))

In [9]:
def get_index():
    # in real life you might load from disk; here we return the in-mem object
    return index

faiss_index = get_index()
faiss_index

<faiss.swigfaiss_avx2.IndexIDMap2; proxy of <Swig Object of type 'faiss::IndexIDMap2Template< faiss::Index > *' at 0x000002065A05FA20> >

In [10]:
faiss_index.add_with_ids(X, ids)

In [11]:
faiss_index.ntotal

104

In [12]:
faiss_index

<faiss.swigfaiss_avx2.IndexIDMap2; proxy of <Swig Object of type 'faiss::IndexIDMap2Template< faiss::Index > *' at 0x000002065A05FA20> >

In [15]:
ps = faiss.ParameterSpace()

def search(query, topk=5, ef_search=None):
    if ef_search is not None:
        ps.set_index_parameter(index, "efSearch", int(ef_search))  # works through wrappers

    q = model.encode([query], convert_to_numpy=True).astype(np.float32)
    faiss.normalize_L2(q)
    D, I = index.search(q, topk)
    out = []
    for score, pid in zip(D[0], I[0]):
        if pid != -1:
            out.append((float(score), int(pid), doc_db[int(pid)]))
    return out


# try a couple of queries
for q in ["How are the AI models evolving",
          "Apple product announcements",
          "best time to contact customers"]:
    print("\nQuery:", q)
    for s, pid, txt in search(q, topk=5, ef_search=64):
        print(f"  score={s:.3f}  id={pid}  -> {txt}")


Query: How are the AI models evolving
  score=0.673  id=0  -> AI models are getting larger every year
  score=0.497  id=58  -> Agents coordinate multiple AI tasks
  score=0.492  id=71  -> OpenAI advances general-purpose AI
  score=0.484  id=43  -> Ethics in AI ensures responsible innovation
  score=0.449  id=12  -> Normalization helps models converge faster

Query: Apple product announcements
  score=0.499  id=9  -> Apple launches new devices every September
  score=0.305  id=37  -> Customer segmentation improves marketing
  score=0.253  id=34  -> Business intelligence helps decision making
  score=0.253  id=86  -> Evaluation metrics guide improvement
  score=0.253  id=39  -> Personalization improves customer experience

Query: best time to contact customers
  score=0.800  id=4  -> The best time to email customers is afternoon
  score=0.527  id=39  -> Personalization improves customer experience
  score=0.396  id=37  -> Customer segmentation improves marketing
  score=0.252  id=82  ->